# BudgiBot Transaction Categorisation 

In [ ]:
# Insalling pip dependecies
%pip install -r requirements.txt

In [ ]:
# Setting up embedding model
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# Sample training data (labelled user examples)
training_examples = [
    ("Paid rent for August", "Rent"),
    ("Monthly Netflix subscription", "Subscriptions"),
    ("Groceries from Walmart", "Groceries"),
    ("Electricity bill payment", "Utilities"),
    ("Watched a movie at PVR", "Shopping & Entertainment"),
    ("Renewed car insurance", "Insurance"),
    ("Doctor consultation fee", "Health"),
    ("Metro card recharge", "Transport"),
]

# Convert sample data to documents with category label as metadata
from langchain.docstore.document import Document

docs = [Document(page_content=ex[0], metadata={"category": ex[1]}) for ex in training_examples]

In [ ]:
# Splitting of sample data into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
split_docs = text_splitter.split_documents(docs)

In [ ]:
# Load or create FAISS vector store
from langchain.vectorstores import FAISS
import os


if os.path.exists("faiss_store/index.faiss"):
    db = FAISS.load_local(
      "faiss_store", 
      embeddings=embedding_model,
      allow_dangerous_deserialization=True  # explicitly allows loading .pkl safely
    )
    print("Using existing FAISS vector DB")
else:
    db = FAISS.from_documents(docs, embedding_model)
    db.save_local("faiss_store")
    print("Created and saved new FAISS vector DB from training examples")

In [ ]:
# Creating a prompt template that is passed to the RAG chain every time as RAG is stateless
from langchain.prompts import PromptTemplate

SYSTEM_PROMPT = PromptTemplate.from_template("""
You are an AI assistant that classifies short user inputs (descriptions of expenses) into predefined categories.

Your job is to choose the most appropriate category for the given input. The categories are:
- Rent
- Insurance
- Utilities
- Shopping & Entertainment
- Groceries
- Subscriptions
- Transport
- Health

Use the following context as examples:
{context}

Now classify the user's input below and respond ONLY with the category name.

In case you are not sure of the category, prompt the user to provide more information.

User Input: {question}
""")



In [ ]:
# Setup Groq LLM
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY")
)

In [ ]:
# Retrieves relevant documents from a vector store based on a user query, injects them into a prompt template, and sends the completed prompt to an LLM to generate a final answer.
from langchain.chains import RetrievalQA

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=False,
    chain_type_kwargs={"prompt": SYSTEM_PROMPT}
)

In [ ]:
# Creating a global array for storing transaction details
import re
from datetime import datetime
transaction_db = []

In [ ]:
# Fuction to classify input and store in transaction db
def classify_input(input_text, amount=None):
    category = rag_chain.run(input_text).strip()
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    transaction_db.append({
        'datetime': now,
        'input': input_text,
        'category': category,
        'amount': amount
    })
    new_doc = Document(page_content=input_text, metadata={"category": category})
    db.add_documents([new_doc])
    db.save_local("faiss_store")
    print("\nCurrent Transaction DB:")
    for t in transaction_db:
        print(t)
    return category


In [ ]:
# Interactive Chat
from IPython.display import display
import ipywidgets as widgets

def interactive_chat():
    input_box = widgets.Text(
        description='Prompt:',
        placeholder='e.g. Bought shoes from Nike',
        layout=widgets.Layout(width='90%')
    )
    output_area = widgets.Output()

    def on_enter(_):
        user_input = input_box.value.strip()
        input_box.value = ''  # Clear input

        if not user_input:
            return

        category = classify_input(user_input)
        with output_area:
            print(f"Input: {user_input}\n→ Predicted Category: {category}\n")

    input_box.on_submit(on_enter)
    display(input_box, output_area)

interactive_chat()


In [21]:
# Print size of locally stored vector db
import os

db = FAISS.load_local(
      "faiss_store", 
      embeddings=embedding_model,
      allow_dangerous_deserialization=True  # explicitly allows loading .pkl safely
    )

# Print number of vectors
print(f"Number of vectors in FAISS DB: {len(db.index_to_docstore_id)}")

# Function to get size of folder
def get_folder_size(path):
    return sum(
        os.path.getsize(os.path.join(dirpath, filename))
        for dirpath, _, filenames in os.walk(path)
        for filename in filenames
    )

# Calculate and print DB size
size_bytes = get_folder_size("faiss_store")
size_mb = size_bytes / (1024 * 1024)
print(f"FAISS DB size on disk: {size_mb:.2f} MB")

Number of vectors in FAISS DB: 20
FAISS DB size on disk: 0.03 MB


In [ ]:
# # Delete locally stored vector db
# import shutil

# shutil.rmtree("faiss_store")
# print("Local FAISS vector DB deleted.")

# transaction_db = []

In [26]:
for transaction in transaction_db:
    print("Date/Time:", transaction['datetime'])
    print("Input:", transaction['input'])
    print("Category:", transaction['category'])
    print("Amount:", transaction['amount'])
    print("-" * 30)  # Separator for readability